# Advanced Model Evaluation Pipeline
## Walk-Forward CV | Stress Test | SHAP | Vertex AI A/B Deploy

In [ ]:
!pip install google-cloud-aiplatform google-cloud-bigquery lightgbm xgboost shap \
    prophet db-dtypes pyarrow scikit-learn pandas matplotlib -q

In [ ]:
from google.colab import auth
auth.authenticate_user()
from google.cloud import bigquery, aiplatform, storage
import pandas as pd, numpy as np, warnings, os, pickle, json, time
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

PROJECT_ID  = "project-8e2366a6-d3cc-40ee-9de"
DATASET_ID  = "hospital_feature_store"
REGION      = "asia-southeast1"
BUCKET_NAME = f"{PROJECT_ID}-hospital-model"
RANDOM_SEED = 42

bq = bigquery.Client(project=PROJECT_ID)
aiplatform.init(project=PROJECT_ID, location=REGION)
os.makedirs("/content/artifacts", exist_ok=True)

print("Loading feature store...")
fs_df = bq.query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.fs_hospital_weekly`
    WHERE target_occupancy_next_week IS NOT NULL
      AND occ_lag4 IS NOT NULL
    ORDER BY hospital_id, report_date
""").to_dataframe()
fs_df["report_date"] = pd.to_datetime(fs_df["report_date"])
print(f"Loaded {len(fs_df):,} rows | {fs_df['hospital_id'].nunique()} hospitals")

EXCLUDE = [
    "hospital_id","report_date","county_fips","zip_code","county_name",
    "hospital_name","feature_computed_at",
    "target_occupancy_next_week","target_high_strain","occupancy_rate",
]
CATEGORICAL   = ["state","hospital_type","season","disease_season",
                 "healthcare_risk_level","metro_nonmetro_flag","hrr_region"]
FEATURE_COLS  = [c for c in fs_df.columns if c not in EXCLUDE]
NUMERIC_FEATS = [c for c in FEATURE_COLS if c not in CATEGORICAL]
CAT_FEATS     = [c for c in CATEGORICAL   if c in FEATURE_COLS]
print(f"Features: {len(FEATURE_COLS)} ({len(NUMERIC_FEATS)} numeric, {len(CAT_FEATS)} categorical)")

Loading feature store...
Loaded 18,181 rows | 3730 hospitals
Features: 61 (54 numeric, 7 categorical)


In [ ]:
# ================================================================
# WALK-FORWARD (ROLLING WINDOW) CROSS VALIDATION
# ================================================================
# - Window size : TRAIN_WEEKS tuan lien tiep (expanding)
# - Step size   : STEP_WEEKS  moi fold
# - Val size    : VAL_WEEKS   tuan ngay sau train window
#
#  |--- train --------->|<- val ->|
#  |--- train + step ----------->|<- val ->|
# ================================================================
from sklearn.pipeline      import Pipeline
from sklearn.compose       import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute        import SimpleImputer
from sklearn.metrics       import mean_absolute_error, r2_score, roc_auc_score
from xgboost  import XGBRegressor, XGBClassifier
import lightgbm as lgb
from prophet import Prophet

TRAIN_WEEKS = 78
STEP_WEEKS  = 8
VAL_WEEKS   = 8

def prophet_factory():
    return Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False
    )

def walk_forward_prophet(df):
    all_dates = sorted(df["report_date"].unique())
    results = []

    fold = 0
    start_idx = 0

    while (start_idx + TRAIN_WEEKS + VAL_WEEKS) <= len(all_dates):

        t_end = start_idx + TRAIN_WEEKS
        v_end = t_end + VAL_WEEKS

        tr = df[df["report_date"].isin(all_dates[start_idx:t_end])]
        va = df[df["report_date"].isin(all_dates[t_end:v_end])]

        train_ts = (
            tr.groupby("report_date")
              ["target_occupancy_next_week"]
              .mean()
              .reset_index()
              .rename(columns={
                  "report_date":"ds",
                  "target_occupancy_next_week":"y"
              })
        )

        model = prophet_factory()
        model.fit(train_ts)

        future = (
            va[["report_date"]]
            .drop_duplicates()
            .rename(columns={"report_date":"ds"})
        )

        fcst = model.predict(future)

        pred_map = dict(zip(fcst.ds, fcst.yhat))

        pred = (
            va["report_date"]
            .map(pred_map)
            .fillna(fcst.yhat.mean())
            .clip(0,1)
            .values
        )

        y = va["target_occupancy_next_week"].clip(0,1).values

        results.append({
            "fold": fold,
            "MAE": mean_absolute_error(y,pred),
            "RMSE": np.sqrt(np.mean((y-pred)**2)),
            "R2": r2_score(y,pred)
        })

        fold += 1
        start_idx += STEP_WEEKS

    return pd.DataFrame(results)

def make_preprocessor(num_feats, cat_feats):
    return ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("scl", StandardScaler()),
        ]), num_feats),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("enc", OrdinalEncoder(handle_unknown="use_encoded_value",
                                   unknown_value=-1, encoded_missing_value=-1)),
        ]), cat_feats),
    ], remainder="drop")

def walk_forward_cv(model_factory, df, feat_cols, num_feats, cat_feats,
                    train_w=TRAIN_WEEKS, step_w=STEP_WEEKS, val_w=VAL_WEEKS):
    all_dates = sorted(df["report_date"].unique())
    n_dates   = len(all_dates)
    results   = []
    fold = 0; start_idx = 0

    while (start_idx + train_w + val_w) <= n_dates:
        t_end = start_idx + train_w
        v_end = t_end + val_w
        tr = df[df["report_date"].isin(all_dates[start_idx:t_end])]
        va = df[df["report_date"].isin(all_dates[t_end:v_end])]

        if len(tr) < 100 or len(va) < 50:
            start_idx += step_w; continue

        prep = make_preprocessor(num_feats, cat_feats)
        tr_encoded = tr.copy()
        va_encoded = va.copy()
        tr_encoded[cat_feats] = tr_encoded[cat_feats].astype(str).replace('None', np.nan).replace('nan', np.nan)
        va_encoded[cat_feats] = va_encoded[cat_feats].astype(str).replace('None', np.nan).replace('nan', np.nan)
        # ----------------------------------------

        Xtr  = prep.fit_transform(tr_encoded[feat_cols]) # Đổi từ tr sang tr_encoded
        Xva  = prep.transform(va_encoded[feat_cols])   # Đổi từ va sang va_encoded
        # Xtr  = prep.fit_transform(tr[feat_cols])
        # Xva  = prep.transform(va[feat_cols])
        ytr  = tr["target_occupancy_next_week"].clip(0,1).values
        yva  = va["target_occupancy_next_week"].clip(0,1).values

        model = model_factory()
        model.fit(Xtr, ytr)
        pred  = model.predict(Xva).clip(0,1)

        results.append({
            "fold": fold,
            "val_start": all_dates[t_end],
            "val_end":   all_dates[v_end-1],
            "n_train": len(tr), "n_val": len(va),
            "MAE":  round(float(mean_absolute_error(yva, pred)), 4),
            "RMSE": round(float(np.sqrt(np.mean((yva-pred)**2))), 4),
            "R2":   round(float(r2_score(yva, pred)), 4),
        })
        fold += 1; start_idx += step_w

    return pd.DataFrame(results)

xgb_factory = lambda: XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    random_state=RANDOM_SEED, n_jobs=-1, verbosity=0)

lgb_factory = lambda: lgb.LGBMRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
    random_state=RANDOM_SEED, n_jobs=-1, verbose=-1)

print("Walk-Forward CV - XGBoost...")
wf_xgb = walk_forward_cv(xgb_factory, fs_df, FEATURE_COLS, NUMERIC_FEATS, CAT_FEATS)
print(f"  {len(wf_xgb)} folds | MAE={wf_xgb.MAE.mean():.4f} +/-{wf_xgb.MAE.std():.4f} | R2={wf_xgb.R2.mean():.4f}")

print("Walk-Forward CV - LightGBM...")
wf_lgb = walk_forward_cv(lgb_factory, fs_df, FEATURE_COLS, NUMERIC_FEATS, CAT_FEATS)
print(f"  {len(wf_lgb)} folds | MAE={wf_lgb.MAE.mean():.4f} +/-{wf_lgb.MAE.std():.4f} | R2={wf_lgb.R2.mean():.4f}")
print("Walk-Forward CV - Prophet...")
wf_prophet = walk_forward_prophet(fs_df)

print(
    f"MAE={wf_prophet.MAE.mean():.4f} "
    f"+/- {wf_prophet.MAE.std():.4f} "
    f"R2={wf_prophet.R2.mean():.4f}"
)

fig, axes = plt.subplots(1, 3, figsize=(15,4))
fig.suptitle("Walk-Forward CV Stability (lower std = more stable)", fontsize=11)
for ax, m in zip(axes, ["MAE","RMSE","R2"]):
    ax.plot(wf_xgb.fold, wf_xgb[m], "o-", color="#2E75B6", lw=1.5, ms=4, label="XGBoost")
    ax.plot(wf_lgb.fold, wf_lgb[m], "s-", color="#1D9E75", lw=1.5, ms=4, label="LightGBM")
    ax.axhline(wf_xgb[m].mean(), color="#2E75B6", ls="--", alpha=0.35)
    ax.axhline(wf_lgb[m].mean(), color="#1D9E75", ls="--", alpha=0.35)
    ax.set_title(m); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/content/artifacts/walkforward_cv.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: walkforward_cv.png")

Walk-Forward CV - XGBoost...
  13 folds | MAE=0.0712 +/-0.0076 | R2=0.7947
Walk-Forward CV - LightGBM...
  13 folds | MAE=0.0714 +/-0.0074 | R2=0.7956
Walk-Forward CV - Prophet...
MAE=0.1883 +/- 0.0042 R2=-0.0065
Saved: walkforward_cv.png


In [ ]:
# ================================================================
# STRESS TEST - isolate high-volatility periods
# ================================================================
# 3 windows:
#   1. COVID Peak   : 2021-10 to 2022-03 (Delta + Omicron)
#   2. Flu Season   : 2022-11 to 2023-03
#   3. Post-COVID   : 2022-07 to 2022-12 (rapid demand rebound)
#
# Protocol: train on ALL data BEFORE window, test ON window.
# Measure MAE vs normal + early-warning hit rate.
# ================================================================

STRESS_WINDOWS = {
    "COVID Peak (Delta+Omicron)": (
        pd.Timestamp("2021-10-01"), pd.Timestamp("2022-03-31")),
    "Flu Season 2022-23": (
        pd.Timestamp("2022-11-01"), pd.Timestamp("2023-03-31")),
    "Post-COVID Rebound": (
        pd.Timestamp("2022-07-01"), pd.Timestamp("2022-12-31")),
}
STRAIN_THRESHOLD = 0.85

def run_stress_test(model_factory, df, feat_cols, num_feats, cat_feats,
                    stress_start, stress_end, name):
    tr = df[df["report_date"] < stress_start]
    te = df[(df["report_date"] >= stress_start) & (df["report_date"] <= stress_end)]
    if len(tr) < 200 or len(te) < 30:
        return None
    prep = make_preprocessor(num_feats, cat_feats)
    Xtr  = prep.fit_transform(tr[feat_cols])
    Xte  = prep.transform(te[feat_cols])
    ytr  = tr["target_occupancy_next_week"].clip(0,1).values
    yte  = te["target_occupancy_next_week"].clip(0,1).values
    model = model_factory(); model.fit(Xtr, ytr)
    pred  = model.predict(Xte).clip(0,1)

    actual_high = yte >= STRAIN_THRESHOLD
    pred_high   = pred >= STRAIN_THRESHOLD
    ew_hit  = float((pred_high & actual_high).sum() / max(actual_high.sum(),1))
    fa_rate = float((pred_high & ~actual_high).sum() / max((~actual_high).sum(),1))

    return {
        "window": name,
        "n_train": len(tr), "n_test": len(te),
        "MAE":  round(float(mean_absolute_error(yte,pred)),4),
        "RMSE": round(float(np.sqrt(np.mean((yte-pred)**2))),4),
        "R2":   round(float(r2_score(yte,pred)),4),
        "early_warning_hit": round(ew_hit,3),
        "false_alarm_rate":  round(fa_rate,3),
        "actual_high_pct":   round(float(actual_high.mean()),3),
    }

stress_xgb_rows, stress_lgb_rows = [], []
for name, (s, e) in STRESS_WINDOWS.items():
    rx = run_stress_test(xgb_factory, fs_df, FEATURE_COLS, NUMERIC_FEATS, CAT_FEATS, s, e, name)
    rl = run_stress_test(lgb_factory, fs_df, FEATURE_COLS, NUMERIC_FEATS, CAT_FEATS, s, e, name)
    if rx: stress_xgb_rows.append(rx)
    if rl: stress_lgb_rows.append(rl)
    if rx and rl:
        print(f"{name[:32]}")
        print(f"  XGB  MAE={rx['MAE']} R2={rx['R2']} EW_hit={rx['early_warning_hit']} FA={rx['false_alarm_rate']}")
        print(f"  LGB  MAE={rl['MAE']} R2={rl['R2']} EW_hit={rl['early_warning_hit']} FA={rl['false_alarm_rate']}")

stress_xgb = pd.DataFrame(stress_xgb_rows)
stress_lgb = pd.DataFrame(stress_lgb_rows)

fig, axes = plt.subplots(1, 2, figsize=(13,5))
fig.suptitle("Stress Test: Volatile Periods vs Normal Baseline", fontsize=11)
labels = stress_xgb["window"].str[:26].tolist()
x = np.arange(len(labels)); w = 0.35

ax = axes[0]
ax.bar(x-w/2, stress_xgb.MAE, w, label="XGBoost",  color="#2E75B6", alpha=0.85)
ax.bar(x+w/2, stress_lgb.MAE, w, label="LightGBM", color="#1D9E75", alpha=0.85)
ax.axhline(wf_xgb.MAE.mean(), color="#2E75B6", ls="--", alpha=0.5, label="XGB normal")
ax.axhline(wf_lgb.MAE.mean(), color="#1D9E75", ls="--", alpha=0.5, label="LGB normal")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=18, ha="right", fontsize=8)
ax.set_title("MAE: Stress vs Normal"); ax.set_ylabel("MAE"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.bar(x-w/2, stress_xgb.early_warning_hit*100, w, label="XGBoost",  color="#2E75B6", alpha=0.85)
ax.bar(x+w/2, stress_lgb.early_warning_hit*100, w, label="LightGBM", color="#1D9E75", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=18, ha="right", fontsize=8)
ax.set_title("Early Warning Hit Rate (%)"); ax.set_ylabel("%"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/artifacts/stress_test.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: stress_test.png")

COVID Peak (Delta+Omicron)
  XGB  MAE=0.0934 R2=0.6734 EW_hit=0.629 FA=0.064
  LGB  MAE=0.0929 R2=0.6717 EW_hit=0.632 FA=0.07
Flu Season 2022-23
  XGB  MAE=0.0765 R2=0.7779 EW_hit=0.724 FA=0.068
  LGB  MAE=0.076 R2=0.7789 EW_hit=0.748 FA=0.069
Post-COVID Rebound
  XGB  MAE=0.0782 R2=0.7531 EW_hit=0.719 FA=0.069
  LGB  MAE=0.0781 R2=0.7535 EW_hit=0.709 FA=0.072
Saved: stress_test.png


In [ ]:
# ================================================================
# TRAIN FINAL MODELS ON FULL TEMPORAL TRAIN SET
# ================================================================
TRAIN_END = "2022-12-31"
VAL_END   = "2023-09-30"

train_df = fs_df[fs_df["report_date"] <= TRAIN_END].copy()
val_df   = fs_df[(fs_df["report_date"] > TRAIN_END) & (fs_df["report_date"] <= VAL_END)].copy()
test_df  = fs_df[fs_df["report_date"] > VAL_END].copy()

prep_final = make_preprocessor(NUMERIC_FEATS, CAT_FEATS)
X_train = prep_final.fit_transform(train_df[FEATURE_COLS])
X_val   = prep_final.transform(val_df[FEATURE_COLS])
X_test  = prep_final.transform(test_df[FEATURE_COLS])
ALL_FEAT_NAMES = NUMERIC_FEATS + CAT_FEATS

ytr_r = train_df["target_occupancy_next_week"].clip(0,1).values
yva_r = val_df  ["target_occupancy_next_week"].clip(0,1).values
yte_r = test_df ["target_occupancy_next_week"].clip(0,1).values
ytr_c = train_df["target_high_strain"].values.astype(int)
yva_c = val_df  ["target_high_strain"].values.astype(int)
yte_c = test_df ["target_high_strain"].values.astype(int)

# XGBoost
pos_w = (ytr_c==0).sum() / max((ytr_c==1).sum(),1)
xgb_reg = XGBRegressor(n_estimators=300,max_depth=6,learning_rate=0.05,
    subsample=0.8,colsample_bytree=0.8,min_child_weight=5,
    random_state=RANDOM_SEED,n_jobs=-1,verbosity=0)
xgb_reg.fit(X_train,ytr_r,eval_set=[(X_val,yva_r)],verbose=False)

xgb_cls = XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.05,
    subsample=0.8,colsample_bytree=0.8,scale_pos_weight=pos_w,eval_metric="auc",
    random_state=RANDOM_SEED,n_jobs=-1,verbosity=0)
xgb_cls.fit(X_train,ytr_c,eval_set=[(X_val,yva_c)],verbose=False)

# LightGBM
from lightgbm import LGBMClassifier
lgb_reg = lgb.LGBMRegressor(n_estimators=300,max_depth=6,learning_rate=0.05,
    subsample=0.8,colsample_bytree=0.8,min_child_samples=20,
    random_state=RANDOM_SEED,n_jobs=-1,verbose=-1)
lgb_reg.fit(X_train,ytr_r,eval_set=[(X_val,yva_r)],
    callbacks=[lgb.early_stopping(30,verbose=False)])

lgb_cls = LGBMClassifier(n_estimators=300,max_depth=6,learning_rate=0.05,
    subsample=0.8,colsample_bytree=0.8,min_child_samples=20,
    class_weight="balanced",random_state=RANDOM_SEED,n_jobs=-1,verbose=-1)
lgb_cls.fit(X_train,ytr_c,eval_set=[(X_val,yva_c)],
    callbacks=[lgb.early_stopping(30,verbose=False)])

print(f"Train:{len(train_df):,} | Val:{len(val_df):,} | Test:{len(test_df):,}")
print("Both models trained.")

Train:9,553 | Val:6,034 | Test:2,594
Both models trained.


In [ ]:
# ================================================================
# MODEL COMPARISON + TRADE-OFF ANALYSIS
# ================================================================
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    precision_recall_curve,
    roc_curve
)

def eval_reg(name, model, X, y):
    p   = model.predict(X).clip(0,1)
    mae = mean_absolute_error(y,p)
    rmse= float(np.sqrt(np.mean((y-p)**2)))
    r2  = r2_score(y,p)
    mape= float(np.mean(np.abs((y-p)/np.clip(y,0.01,None)))*100)
    cov = float(np.mean(np.abs(y-p)<=0.05))
    return {"Model":name,"MAE":round(mae,4),"RMSE":round(rmse,4),
            "R2":round(r2,4),"MAPE%":round(mape,2),"Coverage+-5pp":round(cov,3)}

def eval_cls(name, model, X, y):

    prob = model.predict_proba(X)[:,1]
    pred = model.predict(X)

    return {
        "Model": name,

        "Accuracy": round(
            accuracy_score(y, pred),
            4
        ),

        "ROC-AUC": round(
            roc_auc_score(y, prob),
            4
        ),

        "F1": round(
            f1_score(y, pred, zero_division=0),
            4
        ),

        "Precision": round(
            precision_score(y, pred, zero_division=0),
            4
        ),

        "Recall": round(
            recall_score(y, pred, zero_division=0),
            4
        )
    }

reg_df = pd.DataFrame([eval_reg("XGBoost",xgb_reg,X_test,yte_r),
                        eval_reg("LightGBM",lgb_reg,X_test,yte_r)])
cls_df = pd.DataFrame([eval_cls("XGBoost",xgb_cls,X_test,yte_c),
                        eval_cls("LightGBM",lgb_cls,X_test,yte_c)])

print("="*60); print("REGRESSION"); print("="*60)
print(reg_df.to_string(index=False))
print("\n"+"="*60); print("CLASSIFICATION"); print("="*60)
print(cls_df.to_string(index=False))
print("\n"+"="*60); print("WALK-FORWARD STABILITY"); print("="*60)
stab = pd.DataFrame({
    "Metric": ["MAE mean", "MAE std", "R2 mean", "R2 std"],
    "XGBoost": [
        round(wf_xgb.MAE.mean(), 4),
        round(wf_xgb.MAE.std(), 4),
        round(wf_xgb.R2.mean(), 4),
        round(wf_xgb.R2.std(), 4)
    ],
    "LightGBM": [
        round(wf_lgb.MAE.mean(), 4),
        round(wf_lgb.MAE.std(), 4),
        round(wf_lgb.R2.mean(), 4),
        round(wf_lgb.R2.std(), 4)
    ]
})
print(stab.to_string(index=False))


# ── Radar trade-off chart ────────────────────────────────────────────────
xgb_ew = stress_xgb.early_warning_hit.mean() if len(stress_xgb) else 0.5
lgb_ew = stress_lgb.early_warning_hit.mean() if len(stress_lgb) else 0.5
xgb_stab = 1-wf_xgb.MAE.std()/(wf_xgb.MAE.std()+wf_lgb.MAE.std()+1e-9)
lgb_stab = 1-wf_lgb.MAE.std()/(wf_xgb.MAE.std()+wf_lgb.MAE.std()+1e-9)

def nrm(v, best, worst): return float(np.clip((v-worst)/(best-worst+1e-9),0,1))

xm=reg_df[reg_df.Model=="XGBoost"]["MAE"].values[0]
lm=reg_df[reg_df.Model=="LightGBM"]["MAE"].values[0]
xr=reg_df[reg_df.Model=="XGBoost"]["R2"].values[0]
lr=reg_df[reg_df.Model=="LightGBM"]["R2"].values[0]
xa=cls_df[cls_df.Model=="XGBoost"]["ROC-AUC"].values[0]
la=cls_df[cls_df.Model=="LightGBM"]["ROC-AUC"].values[0]

DIMS = ["MAE (inv)","R2","ROC-AUC","EW Hit","Stability"]
xscores = [nrm(min(xm,lm)-xm+max(xm,lm),max(xm,lm),min(xm,lm)),
           nrm(xr,max(xr,lr),min(xr,lr)),
           nrm(xa,max(xa,la),min(xa,la)),
           nrm(xgb_ew,max(xgb_ew,lgb_ew),min(xgb_ew,lgb_ew)),xgb_stab]
lscores = [nrm(min(xm,lm)-lm+max(xm,lm),max(xm,lm),min(xm,lm)),
           nrm(lr,max(xr,lr),min(xr,lr)),
           nrm(la,max(xa,la),min(xa,la)),
           nrm(lgb_ew,max(xgb_ew,lgb_ew),min(xgb_ew,lgb_ew)),lgb_stab]

angles = np.linspace(0,2*np.pi,len(DIMS),endpoint=False).tolist()
angles += angles[:1]
xscores_p = xscores+xscores[:1]
lscores_p = lscores+lscores[:1]

fig, axes = plt.subplots(1,2,figsize=(14,5),subplot_kw=dict(projection="polar"))
for ax,sc,label,col in [(axes[0],xscores_p,"XGBoost","#2E75B6"),
                         (axes[1],lscores_p,"LightGBM","#1D9E75")]:
    ax.plot(angles,sc,"o-",lw=2,color=col); ax.fill(angles,sc,alpha=0.25,color=col)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(DIMS,size=9)
    ax.set_ylim(0,1); ax.set_title(label,size=12,pad=14); ax.grid(True,alpha=0.3)
fig.suptitle("Trade-off Radar: XGBoost vs LightGBM",fontsize=12)
plt.tight_layout()
plt.savefig("/content/artifacts/tradeoff_radar.png",dpi=130,bbox_inches="tight")
plt.show()

# ── ROC + PR curves ──────────────────────────────────────────────────────
fig,axes = plt.subplots(1,2,figsize=(12,4.5))
for name,cls_m,col in [("XGBoost",xgb_cls,"#2E75B6"),("LightGBM",lgb_cls,"#1D9E75")]:
    prob = cls_m.predict_proba(X_test)[:,1]
    fpr,tpr,_ = roc_curve(yte_c,prob)
    axes[0].plot(fpr,tpr,label=f"{name} AUC={roc_auc_score(yte_c,prob):.3f}",color=col,lw=1.8)
    pv,rv,_ = precision_recall_curve(yte_c,prob)
    axes[1].plot(rv,pv,label=name,color=col,lw=1.8)
axes[0].plot([0,1],[0,1],"--",color="gray",alpha=0.5)
axes[0].set_title("ROC Curve"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].legend(); axes[0].grid(True,alpha=0.3)
axes[1].set_title("Precision-Recall Curve")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].legend(); axes[1].grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig("/content/artifacts/roc_pr_curves.png",dpi=130,bbox_inches="tight")
plt.show()
print("Trade-off charts saved")

REGRESSION
   Model    MAE   RMSE     R2  MAPE%  Coverage+-5pp
 XGBoost 0.0685 0.0944 0.8230  20.64          0.491
LightGBM 0.0675 0.0926 0.8297  19.93          0.498

CLASSIFICATION
   Model  Accuracy  ROC-AUC     F1  Precision  Recall
 XGBoost    0.8921   0.9466 0.7944     0.7696  0.8209
LightGBM    0.8894   0.9460 0.7937     0.7541  0.8376

WALK-FORWARD STABILITY
  Metric  XGBoost  LightGBM
MAE mean   0.0712    0.0714
 MAE std   0.0076    0.0074
 R2 mean   0.7947    0.7956
  R2 std   0.0448    0.0445
Trade-off charts saved


In [ ]:
# ================================================================
# SHAP EXPLAINABILITY (TreeExplainer - O(n) for tree models)
# ================================================================
import shap

X_test_df = pd.DataFrame(X_test, columns=ALL_FEAT_NAMES)

print("Computing SHAP - XGBoost...")
exp_xgb   = shap.TreeExplainer(xgb_reg)
sv_xgb    = exp_xgb.shap_values(X_test_df)
print(f"  SHAP matrix: {sv_xgb.shape}")

print("Computing SHAP - LightGBM...")
exp_lgb   = shap.TreeExplainer(lgb_reg)
sv_lgb    = exp_lgb.shap_values(X_test_df)

# ── Beeswarm summary plots ───────────────────────────────────────────────
fig, axes = plt.subplots(1,2,figsize=(16,7))
for ax,sv,title in [(axes[0],sv_xgb,"XGBoost"),(axes[1],sv_lgb,"LightGBM")]:
    plt.sca(ax)
    shap.summary_plot(sv,X_test_df,max_display=15,show=False,plot_type="dot")
    ax.set_title(f"SHAP Summary - {title}",fontsize=11)
plt.tight_layout()
plt.savefig("/content/artifacts/shap_summary.png",dpi=130,bbox_inches="tight")
plt.show()

# ── Waterfall plots (3 samples: low/medium/high strain) ─────────────────
high_strain_idx = np.where(yte_c == 1)[0]
low_strain_idx  = np.where(yte_c == 0)[0]
SAMPLES = {
    "Low strain"  : int(low_strain_idx[0])  if len(low_strain_idx)  else 0,
    "Borderline"  : int(low_strain_idx[len(low_strain_idx)//2]) if len(low_strain_idx) else 100,
    "High strain" : int(high_strain_idx[0]) if len(high_strain_idx) else 500,
}

fig_wf, axes_wf = plt.subplots(3,2,figsize=(16,13))
fig_wf.suptitle("SHAP Waterfall - Top 10 features per prediction",fontsize=11)

for row_i, (label, idx) in enumerate(SAMPLES.items()):
    for col_i, (sv,base_raw,mname) in enumerate([
        (sv_xgb, exp_xgb.expected_value, "XGBoost"),
        (sv_lgb, exp_lgb.expected_value, "LightGBM"),
    ]):
        ax = axes_wf[row_i, col_i]
        base = float(base_raw) if np.ndim(base_raw)==0 else float(base_raw[0])
        sample_sv = sv[idx]
        top10 = np.argsort(np.abs(sample_sv))[::-1][:10]
        vals  = sample_sv[top10]
        names = [ALL_FEAT_NAMES[i] for i in top10]
        colors= ["#E24B4A" if v>0 else "#2E75B6" for v in vals]
        y_pos = np.arange(len(vals))
        ax.barh(y_pos, vals, color=colors, alpha=0.85)
        ax.set_yticks(y_pos); ax.set_yticklabels(names, fontsize=8)
        pred_v = base + sample_sv.sum()
        true_v = yte_r[idx]
        ax.set_title(
            f"{label} | {mname}  "
            f"base={base:.3f}  pred={pred_v:.3f}  true={true_v:.3f}",
            fontsize=9)
        ax.axvline(0, color="black", lw=0.5); ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("/content/artifacts/shap_waterfall.png",dpi=130,bbox_inches="tight")
plt.show()
print("SHAP charts saved")

Computing SHAP - XGBoost...
  SHAP matrix: (2594, 61)
Computing SHAP - LightGBM...
SHAP charts saved


In [ ]:
# ================================================================
# POST-PROCESSING LAYER
# Each prediction row enriched with Top-3 SHAP features
# Output: BigQuery table ready for Looker Studio waterfall widget
# ================================================================

def build_pred_records(X_sc, y_true_r, y_true_c, model_r, model_c,
                        sv, base_raw, feat_names,
                        hosp_ids, dates, model_name):
    pred_r = model_r.predict(X_sc).clip(0,1)
    pred_c = model_c.predict(X_sc)
    prob_c = model_c.predict_proba(X_sc)[:,1]
    base   = float(base_raw) if np.ndim(base_raw)==0 else float(base_raw[0])
    rows   = []
    for i in range(len(X_sc)):
        top3 = np.argsort(np.abs(sv[i]))[::-1][:3]
        arrow = lambda v: ("up" if v>0 else "down")
        rows.append({
            "model_name"              : model_name,
            "hospital_id"             : str(hosp_ids[i]),
            "report_date"             : str(dates[i])[:10],
            "pred_occupancy_next_week": round(float(pred_r[i]),4),
            "pred_high_strain"        : int(pred_c[i]),
            "pred_high_strain_prob"   : round(float(prob_c[i]),4),
            "actual_occupancy"        : round(float(y_true_r[i]),4),
            "actual_high_strain"      : int(y_true_c[i]),
            "abs_error"               : round(float(abs(y_true_r[i]-pred_r[i])),4),
            "shap_base_value"         : round(base,4),
            "top1_feature"            : feat_names[top3[0]],
            "top1_shap"               : round(float(sv[i][top3[0]]),4),
            "top1_direction"          : arrow(sv[i][top3[0]]),
            "top2_feature"            : feat_names[top3[1]],
            "top2_shap"               : round(float(sv[i][top3[1]]),4),
            "top2_direction"          : arrow(sv[i][top3[1]]),
            "top3_feature"            : feat_names[top3[2]],
            "top3_shap"               : round(float(sv[i][top3[2]]),4),
            "top3_direction"          : arrow(sv[i][top3[2]]),
            "explanation"             : (
                f"Driven by {feat_names[top3[0]]} "
                f"({arrow(sv[i][top3[0]])} {abs(sv[i][top3[0]]):.3f}), "
                f"{feat_names[top3[1]]} "
                f"({arrow(sv[i][top3[1]])} {abs(sv[i][top3[1]]):.3f}), "
                f"{feat_names[top3[2]]} "
                f"({arrow(sv[i][top3[2]])} {abs(sv[i][top3[2]]):.3f})"
            ),
        })
    return pd.DataFrame(rows)

pred_xgb_df = build_pred_records(
    X_test, yte_r, yte_c, xgb_reg, xgb_cls,
    sv_xgb, exp_xgb.expected_value, ALL_FEAT_NAMES,
    test_df["hospital_id"].values, test_df["report_date"].values, "XGBoost")

pred_lgb_df = build_pred_records(
    X_test, yte_r, yte_c, lgb_reg, lgb_cls,
    sv_lgb, exp_lgb.expected_value, ALL_FEAT_NAMES,
    test_df["hospital_id"].values, test_df["report_date"].values, "LightGBM")

all_preds = pd.concat([pred_xgb_df, pred_lgb_df], ignore_index=True)
print(f"Prediction records: {len(all_preds):,}")

# Sample high-strain explanation
hs_sample = all_preds[(all_preds.model_name=="XGBoost") & (all_preds.pred_high_strain==1)].head(2)
if len(hs_sample):
    print("\nHigh-strain prediction example:")
    for _, r in hs_sample.iterrows():
        print(f"  Hospital {r.hospital_id} | pred={r.pred_occupancy_next_week:.1%}")
        print(f"  {r.explanation}")

# Upload to BigQuery
# bq.load_table_from_dataframe(
#     all_preds,
#     f"{PROJECT_ID}.hospital_feature_store.model_predictions_shap",
#     job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
# ).result()
# print("\nUploaded: hospital_feature_store.model_predictions_shap")
# print("Columns available for Looker Studio:")
print([c for c in all_preds.columns])

Prediction records: 5,188

High-strain prediction example:
  Hospital 010039 | pred=92.1%
  Driven by occ_roll4 (up 0.164), occ_lag1 (up 0.049), occ_roll8 (up 0.029)
  Hospital 010087 | pred=88.9%
  Driven by occ_roll4 (up 0.165), occ_lag1 (up 0.033), occ_roll8 (up 0.032)
['model_name', 'hospital_id', 'report_date', 'pred_occupancy_next_week', 'pred_high_strain', 'pred_high_strain_prob', 'actual_occupancy', 'actual_high_strain', 'abs_error', 'shap_base_value', 'top1_feature', 'top1_shap', 'top1_direction', 'top2_feature', 'top2_shap', 'top2_direction', 'top3_feature', 'top3_shap', 'top3_direction', 'explanation']


In [ ]:
# ================================================================
# SAVE ARTIFACTS + VERTEX AI: DEPLOY 2 MODELS A/B
# ================================================================
CONTAINER = "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest"

def save_and_upload(tag, model_r, model_c, prep, explainer, gcs_prefix):
    d = f"/content/artifacts/{tag}"
    os.makedirs(d, exist_ok=True)
    for fname, obj in [("model_reg.pkl",model_r),("model_cls.pkl",model_c),
                        ("preprocessor.pkl",prep),("explainer.pkl",explainer)]:
        with open(f"{d}/{fname}","wb") as f: pickle.dump(obj,f)

    predict_src = (
        "import pickle,os,numpy as np,pandas as pd\n"
        "_a={}\n"
        "def _load():\n"
        "    b=os.environ.get('AIP_STORAGE_URI','/artifacts/TAG')\n"
        "    for n in ['model_reg','model_cls','preprocessor','explainer']:\n"
        "        with open(f'{b}/{n}.pkl','rb') as f: _a[n]=pickle.load(f)\n"
        "def predict(instances,**kw):\n"
        "    if not _a: _load()\n"
        "    df=pd.DataFrame(instances)\n"
        "    X=_a['preprocessor'].transform(df)\n"
        "    occ=_a['model_reg'].predict(X).clip(0,1)\n"
        "    cls=_a['model_cls'].predict(X)\n"
        "    prob=_a['model_cls'].predict_proba(X)[:,1]\n"
        "    sv=_a['explainer'].shap_values(pd.DataFrame(X,columns=list(df.columns)[:X.shape[1]]))\n"
        "    out=[]\n"
        "    for i in range(len(occ)):\n"
        "        t3=list(np.argsort(np.abs(sv[i]))[::-1][:3].astype(int))\n"
        "        out.append({'occupancy_next_week':round(float(occ[i]),4),\n"
        "            'high_strain':int(cls[i]),'prob':round(float(prob[i]),4),\n"
        "            'top_features':[{'f':df.columns[j],'shap':round(float(sv[i][j]),4)} for j in t3]})\n"
        "    return out\n"
    ).replace("TAG", tag)
    with open(f"{d}/predict.py","w") as f: f.write(predict_src)

    # upload to GCS
    sc = storage.Client(project=PROJECT_ID)
    bkt = sc.bucket(BUCKET_NAME)
    for fn in os.listdir(d):
        bkt.blob(f"{gcs_prefix}/{fn}").upload_from_filename(f"{d}/{fn}")
    gcs_uri = f"gs://{BUCKET_NAME}/{gcs_prefix}"
    print(f"  [{tag}] uploaded -> {gcs_uri}")
    return gcs_uri

# Ensure bucket exists
sc = storage.Client(project=PROJECT_ID)
try:
    sc.create_bucket(BUCKET_NAME, location=REGION)
except Exception: pass

gcs_xgb = save_and_upload("xgboost",  xgb_reg, xgb_cls, prep_final, exp_xgb, "hospital-model/xgboost")
gcs_lgb = save_and_upload("lightgbm", lgb_reg, lgb_cls, prep_final, exp_lgb, "hospital-model/lightgbm")

# Register in Model Registry
m_xgb = aiplatform.Model.upload(
    display_name="hospital-xgboost-v1", artifact_uri=gcs_xgb,
    serving_container_image_uri=CONTAINER,
    serving_container_environment_variables={"AIP_STORAGE_URI": gcs_xgb})
m_lgb = aiplatform.Model.upload(
    display_name="hospital-lightgbm-v1", artifact_uri=gcs_lgb,
    serving_container_image_uri=CONTAINER,
    serving_container_environment_variables={"AIP_STORAGE_URI": gcs_lgb})
print(f"XGBoost  registered: {m_xgb.resource_name}")
print(f"LightGBM registered: {m_lgb.resource_name}")

# Create endpoint + deploy both with 50/50 traffic
endpoint = aiplatform.Endpoint.create(display_name="hospital-ab-endpoint")
print("Deploying XGBoost (may take ~10 min)...")
dm_xgb = endpoint.deploy(model=m_xgb, deployed_model_display_name="xgboost-v1",
    machine_type="n1-standard-2", min_replica_count=1, max_replica_count=2,
    traffic_split={"0": 100})

print("Deploying LightGBM and setting 50/50 split...")
dm_lgb = endpoint.deploy(model=m_lgb, deployed_model_display_name="lightgbm-v1",
    machine_type="n1-standard-2", min_replica_count=1, max_replica_count=2,
    traffic_split={"0": 0})
endpoint.update_traffic_split({dm_xgb.id: 50, dm_lgb.id: 50})
print(f"Endpoint: {endpoint.resource_name}")
print("Traffic: XGBoost 50% | LightGBM 50%")

with open("/content/artifacts/endpoint_ab_info.json","w") as f:
    json.dump({"endpoint_id": endpoint.name,
               "resource_name": endpoint.resource_name,
               "dm_xgb_id": dm_xgb.id,
               "dm_lgb_id": dm_lgb.id}, f, indent=2)

In [ ]:
# ================================================================
# ONLINE A/B: call endpoint, measure latency, select winner
# ================================================================
import time

sample_req = test_df[FEATURE_COLS].head(20).to_dict(orient="records")
y_sample_r = yte_r[:20]; y_sample_c = yte_c[:20]

t0 = time.time()
resp = endpoint.predict(instances=sample_req)
latency_ms = (time.time()-t0)*1000
print(f"Endpoint latency (20 samples): {latency_ms:.0f} ms")
print(f"Predictions received: {len(resp.predictions)}")

# ── Final composite score ────────────────────────────────────────────────
def composite_score(mae, r2, auc, ew_hit, mae_std):
    # Weights: accuracy 35%, R2 25%, AUC 20%, early-warning 10%, stability 10%
    return 0.35*(1-mae) + 0.25*r2 + 0.20*auc + 0.10*ew_hit + 0.10*(1-mae_std)

xgb_score = composite_score(
    reg_df[reg_df.Model=="XGBoost"]["MAE"].values[0],
    reg_df[reg_df.Model=="XGBoost"]["R2"].values[0],
    cls_df[cls_df.Model=="XGBoost"]["ROC-AUC"].values[0],
    stress_xgb.early_warning_hit.mean() if len(stress_xgb) else 0.5,
    wf_xgb.MAE.std())

lgb_score = composite_score(
    reg_df[reg_df.Model=="LightGBM"]["MAE"].values[0],
    reg_df[reg_df.Model=="LightGBM"]["R2"].values[0],
    cls_df[cls_df.Model=="LightGBM"]["ROC-AUC"].values[0],
    stress_lgb.early_warning_hit.mean() if len(stress_lgb) else 0.5,
    wf_lgb.MAE.std())

winner      = "XGBoost" if xgb_score >= lgb_score else "LightGBM"
winner_dm_id= dm_xgb.id if winner=="XGBoost" else dm_lgb.id
loser_dm_id = dm_lgb.id if winner=="XGBoost" else dm_xgb.id

print(f"\nComposite score:")
print(f"  XGBoost  : {xgb_score:.4f}")
print(f"  LightGBM : {lgb_score:.4f}")
print(f"  Winner   : {winner}")

# Shift 100% traffic to winner
endpoint.update_traffic_split({winner_dm_id: 100, loser_dm_id: 0})
print(f"Traffic updated: 100% -> {winner}")

# ── Summary printout ─────────────────────────────────────────────────────
lines = [
    "DEPLOYMENT SUMMARY",
    "  Walk-Forward CV   -> walkforward_cv.png",
    "  Stress Test       -> stress_test.png",
    "  Trade-off Radar   -> tradeoff_radar.png",
    "  ROC/PR Curves     -> roc_pr_curves.png",
    "  SHAP Summary      -> shap_summary.png",
    "  SHAP Waterfall    -> shap_waterfall.png",
    "  Predictions+SHAP  -> BQ: model_predictions_shap",
    "  Endpoint          -> Vertex AI (winner model active)",
    "  Looker Studio: join model_predictions_shap with fs_hospital_weekly",
    "  Fields: top1/2/3_feature + shap values for waterfall chart",
]
print("
".join(lines))